# Глава 4 — Memory

Краткосрочная память: история диалога, обрезка и суммаризация. RAG добавляет найденные внешние документы в контекст запроса. Все данные этого учебного проекта находятся в RAM и не сохраняются между перезапусками.

Запускайте notebook из корня проекта. Подготовка окружения описана в README. Последний раздел требует отдельно установленной модели `embeddinggemma`.

In [ ]:
from agent import TinyAgent
from llm import LLM, EmbeddingModel
from memory import Memory, TrimmingMemory, SummarizationMemory, RAGMemory
from illustrated_agents.utils import TrajectoryViewer

llm = LLM(model="gemma4:e4b", temperature=0)

## История диалога

При следующем запросе модель получает предыдущие сообщения. Trajectory отдельно хранит шаги выполнения.

In [ ]:
agent = TinyAgent(llm=llm, memory=Memory())
print(agent.run("Hi! My name is Sarah and I live in Lisbon."))
print(agent.run("What is my name and where do I live?"))

In [ ]:
print(agent.memory.get_messages())
print(agent.trajectory.runs)

In [ ]:
TrajectoryViewer(agent.trajectory)

## Обрезка

Сохраняем system-инструкции и последние две реплики пользователя вместе с относящимися к ним ответами. Новая реплика пользователя уже считается одним из двух ходов. В отличие от простого среза последних четырёх сообщений, начало запроса не остаётся без своего user-сообщения.

In [ ]:
memory = TrimmingMemory(max_turns=2)
memory.add("system", "Reply briefly in English.")
trimmed_agent = TinyAgent(llm=llm, memory=memory)
for query in ("My name is Sarah.", "I live in Lisbon.", "What is 1 + 1?", "What is 2 + 2?"):
    print(trimmed_agent.run(query))
print(memory.get_messages())

In [ ]:
TrajectoryViewer(trimmed_agent.trajectory)

## Суммаризация

После каждого ответа выполняется дополнительный вызов LLM для обновления сводки. Исходные system-инструкции сохраняются отдельно. Пустая сводка не удаляет историю; ошибка запроса пробрасывается вызывающему коду, история остаётся доступной. Сводка может потерять детали и не задаёт жёсткого лимита токенов.

In [ ]:
summary_memory = SummarizationMemory(llm=llm)
summary_memory.add("system", "Reply briefly in English.")
summary_agent = TinyAgent(llm=llm, memory=summary_memory)
print(summary_agent.run("My name is Sarah. My favorite animal is a flamingo."))
print(summary_agent.run("What is my name and favorite animal?"))
print(summary_memory.get_messages())

In [ ]:
TrajectoryViewer(summary_agent.trajectory)

## RAG — поиск по внешним документам

Требуется `embeddinggemma` на том же сервере Ollama. Если модель отсутствует, предыдущие разделы остаются работоспособными; этот раздел даст ошибку отсутствующей модели. Здесь документы индексируются при создании памяти, затем выбираются три наиболее похожих по cosine similarity. Порог релевантности, постоянная база данных и agentic RAG в этот учебный пример не входят.

In [ ]:
embedding_model = EmbeddingModel(model="embeddinggemma")
a = embedding_model.embed("I love flamingos.")
b = embedding_model.embed("Dolphins use echolocation.")
c = embedding_model.embed("Flamingos are pink birds.")
print("Vector dimensions:", len(a))
print("Flamingos / dolphins:", RAGMemory._cosine(a, b))
print("Flamingos / flamingos:", RAGMemory._cosine(a, c))

In [ ]:
documents = [
    "Sarah works as a marine biologist studying coral reefs.",
    "Sarah lives in Lisbon, Portugal.",
    "Sarah's favorite hobby is rock climbing.",
    "Sarah's favorite animal is the flamingo.",
    "Sarah speaks Spanish and Portuguese.",
    "Ilse is a software engineer at a renewable energy startup.",
    "Ilse lives in Amsterdam, the Netherlands.",
    "Ilse plays the cello in a local string quartet.",
    "Ilse's favorite author is Brandon Sanderson.",
    "Ilse's favorite animal is the dolphin.",
]
rag_memory = RAGMemory(embedding_model=embedding_model, documents=documents)
query = "What is Sarah's favorite animal?"
print("Retrieved:", rag_memory.search(query))
rag_agent = TinyAgent(llm=llm, memory=rag_memory)
print(rag_agent.run(query))
print(rag_memory.get_messages())

In [ ]:
TrajectoryViewer(rag_agent.trajectory)